# 环节 07 · Block 堆叠与整体架构（配套 Notebook）

> 配套长文：[环节07-Block堆叠与整体架构详解.md](./环节07-Block堆叠与整体架构详解.md)
> 定位：把"Block 堆成模型""RNN/LSTM 为什么被换掉""三种架构怎么推算下一个 token"跑成代码。纯 Python 标准库，零依赖。

| 本 Notebook | 长文章节 | 验证什么 |
|---|---|---|
| §1 堆叠与参数量 | §1 / §1.2 | 形状 (n,d) 全程不变；参数随层数线性涨 |
| §2 RNN 数值例子 | §4.1 | 复现 h1/h2/h3 与 logits |
| §3 RNN 的数学缺陷 | §2.1 | ∏W 连乘 → 梯度指数消失 |
| §4 LSTM 的记忆传送带 | §2.2 | ∂c_t/∂c_{t-1} = f_t，梯度有直达路径 |
| §5 Transformer decode | §4.3 | 复现 z3 与最后的 softmax |
| §6 三种架构的 Mask | §3 | 双向 / 因果 / 交叉注意力的形状差异 |


## 1. 一个 Block 四件套，N 层堆叠（长文 §1）

**"重复 N 层、每层相同"指结构相同，不是参数相同**——每层都有自己独立的一套 `W_Q/W_K/W_V/W_O`、FFN、LN 参数，所以参数量随层数**线性**增长。


In [ ]:
import math


def block_params(d, d_ff, heads_kv_ratio=1.0):
    """一层 Block 的参数：Attention 4d² + FFN 3·d·d_ff（SwiGLU）。"""
    return 4 * d * d + 3 * d * d_ff


configs = [
    ("GPT-2 small", 768, 3072, 12, 12, 50257),
    ("GPT-2 large", 1280, 5120, 36, 20, 50257),
    ("LLaMA-2 7B", 4096, 11008, 32, 32, 32000),
    ("LLaMA-2 70B", 8192, 28672, 80, 64, 32000),
]

print(f"{'模型':<14} {'d':>6} {'层数':>5} {'每层参数':>12} {'N 层合计':>12} {'+Embedding':>12} {'总参数':>10}")
print("-" * 82)
for name, d, d_ff, L, H, V in configs:
    per = block_params(d, d_ff)
    layers = per * L
    emb = V * d
    print(f"{name:<14} {d:>6} {L:>5} {per/1e6:>10.1f} M {layers/1e9:>10.2f} B"
          f" {emb/1e6:>10.1f} M {(layers+emb)/1e9:>8.2f} B")

print("\n→ 参数量 ≈ 每层参数 × 层数 + Embedding，对层数是**线性**的。")
print("  “每层结构相同”但参数独立，所以层与层才能分化出不同分工（浅层词法 → 深层语义）。")
print("\n注：这里 Attention 按 4d²（MHA）估算。真实 LLaMA-2 70B 用 GQA（KV 头少于 Q 头），")
print("    KV/O 投影更小，所以它的实际参数量比上表的 78B 略低（约 69B，口径见环节 04 §3）。")


In [ ]:
# 层间数据流：形状 (n, d) 全程不变
import random

random.seed(0)
n, d, N = 4, 6, 3
x = [[random.gauss(0, 1) for _ in range(d)] for _ in range(n)]
print(f"输入 X：{len(x)} 行 × {len(x[0])} 列   （n 个 token，每个 d 维）")
for layer in range(1, N + 1):
    # 每层只做一次“变换 + 残差”，形状不变（真实内容由 Attention/FFN 决定）
    y = [[v * 0.9 + 0.1 for v in row] for row in x]
    print(f"  第 {layer} 层输出：{len(y)} × {len(y[0])}   形状不变 = {len(y) == len(x) and len(y[0]) == len(x[0])}")
print("\n→ 残差主干 X ← X + 子层(X) 决定了形状必须保持 (n, d)：")
print("  n 行 = n 个 token，每行 = 该 token 目前对上下文的表示；")
print("  变的只是行向量里的语义（每过一层多融合一次前缀信息），容器始终是这个 (n,d)。")


## 2. RNN 的数值例子（长文 §4.1）

设定：h 维 = 2，词表 {a,b,c}，`h_t = tanh(h_{t-1} + W_x·x_t)`（`W_h = I` 简化）。


In [ ]:
x_a, x_b, x_c = [1.0, 0.0], [0.0, 1.0], [1.0, 1.0]
W_x = [[1.0, 0.0], [0.0, 1.0]]


def tanh_v(v):
    return [math.tanh(t) for t in v]


def mv(M, v):
    return [sum(p * q for p, q in zip(row, v)) for row in M]


h1 = tanh_v(mv(W_x, x_a))
h2 = tanh_v([p + q for p, q in zip(h1, mv(W_x, x_b))])
h3 = tanh_v([p + q for p, q in zip(h2, mv(W_x, x_c))])

print(f"h1 = tanh(W_x·x_a)           = [1.0000, 0.0000]  →  {[round(v, 4) for v in h1]}")
print(f"h2 = tanh(h1 + W_x·x_b)      = [0.7616, 1.0000]  →  {[round(v, 4) for v in h2]}")
print(f"h3 = tanh(h2 + W_x·x_c)      = [1.6416, 1.7616]  →  {[round(v, 4) for v in h3]}")
print("（长文 §4.1 写 [0.76, 0.00] / [0.64, 0.76] / [0.93, 0.94]，一致）")

W_o = [[1, 0], [0, 1], [1, 1]]
logits = mv(W_o, h3)


def softmax(v):
    m = max(v)
    e = [math.exp(t - m) for t in v]
    s = sum(e)
    return [t / s for t in e]


print(f"\nlogits = W_o·h3 = {[round(v, 4) for v in logits]}   （长文 [0.93, 0.94, 1.87]）")
print(f"softmax         = {[round(v, 4) for v in softmax(logits)]}")
print(f"argmax = 第 {max(range(3), key=lambda i: logits[i]) + 1} 个词")
print("\n注：长文写 softmax ≈ [0.25, 0.25, 0.50] 是示意值；精确值是上面这个，")
print("    argmax 仍是第 3 个词，结论不变。")


## 3. RNN 的数学缺陷：梯度连乘（长文 §2.1）

`∂L/∂h_1` 要经过 t 次链式连乘，含 `∏ W_hᵀ`。`W_h` 的特征值 < 1 → 梯度指数衰减。


In [ ]:
print("W_h = 0.5·I（特征值 0.5 < 1）时，梯度系数随步数：")
for steps in (5, 10, 20, 50, 100):
    print(f"  传 {steps:>3} 步：0.5^{steps} = {0.5 ** steps:.3e}")
print("\n→ 20 步就掉到 1e-6，100 步基本是 0 —— 这就是“记不住超过 ~10 步”的数学原因。")

print("\n特征值 > 1 的另一边（爆炸），同样致命：")
for steps in (5, 10, 20):
    print(f"  传 {steps:>3} 步：1.5^{steps} = {1.5 ** steps:.3e}")

print("\n→ 所以 RNN 的问题是结构性的：要么衰减要么爆炸，靠调参只能挪区间。")
print("  Transformer 的答案：不压缩历史，每个 token 直接可见（但代价 O(n²)，见环节 04）。")


## 4. LSTM 的记忆传送带（长文 §2.2）

LSTM 的关键是 `c_t = f_t ⊙ c_{t-1} + i_t ⊙ c̃_t`。求导：

```
∂c_t / ∂c_{t-1} = f_t        ← 只剩遗忘门，不经过任何权重矩阵！
```

跟 RNN 的"连乘 W 矩阵"相比，这是一条**直达路径**（和残差异曲同工）。当 `f_t ≈ 1` 时，梯度可以近乎无损地穿越很多步。


In [ ]:
print(f"{'遗忘门 f_t 恒定值':>18} {'传 100 步后梯度系数':>22}")
print("-" * 44)
for f in (0.9, 0.99, 1.0):
    print(f"{f:>18} {f ** 100:>22.4f}")
print("\n→ f_t → 1 时几乎不衰减（对比 RNN 的 0.5^100 ≈ 1e-30）。")
print("  但注意：这只是“能传”，不代表“会记”——门开多大由训练学出来，")
print("  而且历史仍被压进一个固定维度的 c_t，容量瓶颈还在（长文 §2.2 末）。")


## 5. Transformer 怎么推算下一个 token（长文 §4.3）

序列一次并行前向，取**最后一个位置**的向量 → LM Head → 概率分布。


In [ ]:
X3 = [
    [1.0, 1.0, 1.0, 0.0],     # a（pos0）
    [0.0, 1.0, 1.0, 1.0],     # b（pos1）
    [1.0, 1.0, 0.0, 1.0],     # c（pos2）
]
n3, dk3 = 3, 4
S3 = [[sum(X3[i][k] * X3[j][k] for k in range(dk3)) / math.sqrt(dk3) for j in range(n3)]
      for i in range(n3)]
print("Score = Q·Kᵀ/√4：")
for i, row in enumerate(S3):
    print(f"  行{i+1}: {[round(v, 4) for v in row]}")
print("（长文行1 [1.5, 1.0, 1.0] / 行3 [1.0, 1.0, 1.5]，一致）")

P3 = softmax(S3[2])
print(f"\n行3（“c”看前面所有 token）softmax = {[round(v, 4) for v in P3]}")
z3 = [sum(P3[j] * X3[j][k] for j in range(n3)) for k in range(dk3)]
print(f"z3 = {[round(v, 4) for v in z3]}   （长文 [0.726, 1.000, 0.548, 0.726]）")
print("\n→ z3 就是“行3 能看见的三个向量按注意力权重加权求和”，这就是“注意力即上下文”。")

logits_demo = [1.274, 1.726, 1.726, 1.274, 1.452]
print(f"\nLM Head（长文给的示意 logits {logits_demo}）→ softmax =")
print(f"  {[round(v, 4) for v in softmax(logits_demo)]}   （长文 [0.158, 0.248, 0.248, 0.158, 0.188]）")
print("\n注：长文 §4.3 未给出 W_head 的具体矩阵，这几个 logits 是示意值；")
print("    本格验证的是“logits → softmax → 采样”这一步。")


## 6. 三种架构的 Mask 形状（长文 §3）

差别全在那张 `(n, n)` 的注意力掩码上：


In [ ]:
def show_mask(title, name, n, allow):
    print(f"\n{title}")
    for i in range(n):
        print("   " + " ".join("✓" if allow(i, j) else "✗" for j in range(n)))
    print(f"   （{name}）")


N = 5
show_mask("Decoder-only（因果掩码，只看自己和左边）", "下三角", N, lambda i, j: j <= i)
show_mask("Encoder-only（双向，全都能看）", "全 1", N, lambda i, j: True)
show_mask("Encoder–Decoder 的交叉注意力（解码器看完整输入）", "整行可见，但只对源序列",
          N, lambda i, j: True)

print("\n三种结构对照（长文 §3）：")
print(f"{'架构':<18} {'自注意力方向':<16} {'典型任务':<22} 代表")
print("-" * 72)
for name, mask, task, rep in [
    ("Encoder-only", "双向", "理解/分类/检索", "BERT"),
    ("Encoder–Decoder", "编码双向+解码因果+交叉", "翻译/摘要", "T5、BART"),
    ("Decoder-only", "因果（只看过去）", "一切生成（主流）", "GPT/LLaMA/Qwen"),
]:
    print(f"{name:<18} {mask:<16} {task:<22} {rep}")

print("\n→ 为什么 Decoder-only 赢（长文 §3）：一套掩码自回归同时搞定训练与推理，")
print("  不需要交叉注意力 → 结构最简 → 同算力能堆更大规模。")


## 7. 自测（长文 §6）

| 问题 | 本 Notebook 的现场证据 |
|---|---|
| "重复 N 层、每层相同"是什么意思？ | §1：结构相同、参数独立，参数量对层数线性 |
| 为什么层间形状一直是 (n,d)？ | §1 第二个实验：残差要求形状不变 |
| RNN 为什么梯度消失？ | §3：`0.5^20 ≈ 1e-6`，连乘导致指数衰减 |
| LSTM 怎么缓解？ | §4：`∂c_t/∂c_{t-1} = f_t`，不含权重矩阵，是直达路径 |
| Transformer 推理时取哪一行？ | §5：只取最后一个位置的 z_last |
| 三种架构差在哪？ | §6：差别全在 `(n,n)` 掩码上 |

**上一站** [环节 06 · 残差与归一化](./环节06-残差连接与归一化详解.md)   **下一站** [环节 08 · 输出头与训练目标](./环节08-输出头与训练目标详解.md)
